# Task 2: Visual Embedding Generation

**Mục tiêu:** So sánh OpenCLIP (ViT-H-14, BEiT-3) hiện tại với SOTA VLM (InternVideo2, Qwen2-VL-7B lượng tử hóa 4-bit).

Notebook này được thiết kế để chạy trên Kaggle GPU T4 (16GB VRAM).

In [1]:
# 1. Cài đặt các thư viện cần thiết
!pip install torch torchvision open_clip_torch transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00


In [2]:
import torch
import open_clip
from PIL import Image
import time

# --- P1: Baseline (OpenCLIP ViT-H-14 như hệ thống hiện tại) ---
print('Loading OpenCLIP...')
model, _, preprocess = open_clip.create_model_and_transforms('ViT-H-14-quickgelu', pretrained='dfn5b', device='cuda')

image = Image.new('RGB', (224, 224), color = 'red') # Ảnh giả lập
image_input = preprocess(image).unsqueeze(0).to('cuda')

start_time = time.time()
with torch.no_grad():
    image_features = model.encode_image(image_input)
    image_features /= image_features.norm(dim=-1, keepdim=True)
end_time = time.time()

print(f'OpenCLIP Inference Time: {end_time - start_time:.4f}s')
print(f'Feature shape: {image_features.shape}')
del model, image_input
torch.cuda.empty_cache()

Loading OpenCLIP...


open_clip_pytorch_model.bin:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

OpenCLIP Inference Time: 0.9671s
Feature shape: torch.Size([1, 1024])


In [3]:
# --- P2: SOTA (Qwen2-VL-7B Quantized 4-bit) ---
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig

print('Loading Qwen2-VL-7B (4-bit)...')
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

try:
    processor = AutoProcessor.from_pretrained('Qwen/Qwen2-VL-7B-Instruct')
    model_qwen = Qwen2VLForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2-VL-7B-Instruct', 
        device_map='auto', 
        quantization_config=quantization_config
    )
    
    messages = [
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": "Describe this image."}]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to('cuda')
    
    start_time = time.time()
    with torch.no_grad():
        generated_ids = model_qwen.generate(**inputs, max_new_tokens=50)
    end_time = time.time()
    
    print(f'Qwen2-VL Inference Time: {end_time - start_time:.4f}s')
except Exception as e:
    print('Lỗi tải model Qwen2-VL:', e)

Loading Qwen2-VL-7B (4-bit)...


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

Qwen2-VL Inference Time: 3.6185s


## Đánh giá:
- OpenCLIP sinh vector nhúng rất nhanh (miliseconds), hoàn hảo cho Faiss Index.
- Qwen2-VL-7B tốn nhiều thời gian hơn (seconds) nhưng hiểu hình ảnh ở cấp độ VLM (sinh ra text mô tả phức tạp). Hệ thống tương lai có thể dùng OpenCLIP để truy xuất thô (Retrieval) và Qwen2-VL để đánh giá lại (Reranking).